# C2 Function 2 – Supported-Minority Fixed-Budget Selector

This notebook implements the first end-to-end prototype of the **Component 2 constrained review-evidence selector**.

## Goal

For each product, select a compact fixed-budget set of reviews that balances:

- predicted review usefulness,
- semantic representativeness,
- aspect coverage,
- supported-minority preservation,
- semantic redundancy control,
- opinion-distribution fidelity.

The first prototype uses a **two-phase constrained greedy selection strategy**.

## Important Limitation

The final Component 1 aspect/sentiment output is not yet available.

Therefore:

- usefulness scores are derived from real review text,
- Sentence-BERT embeddings are derived from real review text,
- aspect/sentiment fields currently come from clearly labelled synthetic C1 integration-test data,
- supported-minority membership is therefore also provisional,
- minority, coverage, and opinion-fidelity results produced from those mock labels are **not final research findings**.

The review budget, selector weights, priority-group count, and reserved minority slots used in this notebook are provisional prototype settings and must later be tested using sensitivity analysis.

## Evaluation Note

The saved Function 1 usefulness model is applied to all reviews to demonstrate the selector pipeline. Some candidate reviews belong to the period used to train the preliminary Function 1 model.

Therefore, this notebook demonstrates selector functionality rather than providing a final unbiased selector-performance estimate.

Final research evaluation should use appropriately held-out or out-of-fold usefulness scores where required.


In [1]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import jensenshannon

print("Function 2 selector environment ready")


Function 2 selector environment ready


## 1. Resolve Project Paths

This notebook reuses artifacts created by:

- `03_usefulness_baseline.ipynb`
- `04_c1_semantic_representation.ipynb`

All required artifacts are checked before the selector is executed.


In [2]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "component_2").exists():
    COMPONENT_ROOT = CURRENT_DIR / "component_2"

elif CURRENT_DIR.name == "component_2":
    COMPONENT_ROOT = CURRENT_DIR

elif (
    CURRENT_DIR.name == "notebooks"
    and CURRENT_DIR.parent.name == "component_2"
):
    COMPONENT_ROOT = CURRENT_DIR.parent

else:
    raise RuntimeError(
        f"Could not locate component_2 from current directory: {CURRENT_DIR}"
    )


PILOT_PATH = (
    COMPONENT_ROOT
    / "data"
    / "pilot"
    / "smartphone_pilot_cleaned.parquet"
)

MODEL_PATH = (
    COMPONENT_ROOT
    / "models"
    / "tfidf_ridge_usefulness.joblib"
)

VECTORIZER_PATH = (
    COMPONENT_ROOT
    / "models"
    / "tfidf_vectorizer.joblib"
)

INTEGRATION_DIR = (
    COMPONENT_ROOT
    / "results"
    / "integration"
)

EMBEDDING_PATH = (
    INTEGRATION_DIR
    / "smartphone_review_embeddings.npy"
)

EMBEDDING_INDEX_PATH = (
    INTEGRATION_DIR
    / "smartphone_review_embedding_index.csv"
)

C1_MOCK_PATH = (
    INTEGRATION_DIR
    / "c1_mock_aspect_sentiment.csv"
)

SUPPORTED_MINORITY_PATH = (
    INTEGRATION_DIR
    / "mock_supported_minority_groups.csv"
)

SELECTOR_RESULTS_DIR = (
    COMPONENT_ROOT
    / "results"
    / "selector"
)

SELECTOR_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


required_paths = {
    "pilot": PILOT_PATH,
    "usefulness_model": MODEL_PATH,
    "tfidf_vectorizer": VECTORIZER_PATH,
    "embeddings": EMBEDDING_PATH,
    "embedding_index": EMBEDDING_INDEX_PATH,
    "mock_c1": C1_MOCK_PATH,
    "supported_minority_groups": SUPPORTED_MINORITY_PATH,
}


print("Current directory:", CURRENT_DIR)
print("Component root:", COMPONENT_ROOT)

for name, path in required_paths.items():
    print(
        f"{name}: {path.exists()} -> {path}"
    )


missing_paths = [
    str(path)
    for path in required_paths.values()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required selector artifacts:\n"
        + "\n".join(missing_paths)
    )

print(
    "\nAll required selector artifacts found."
)


Current directory: d:\Research\Repository\J26-DS-349\component_2\notebooks
Component root: d:\Research\Repository\J26-DS-349\component_2
pilot: True -> d:\Research\Repository\J26-DS-349\component_2\data\pilot\smartphone_pilot_cleaned.parquet
usefulness_model: True -> d:\Research\Repository\J26-DS-349\component_2\models\tfidf_ridge_usefulness.joblib
tfidf_vectorizer: True -> d:\Research\Repository\J26-DS-349\component_2\models\tfidf_vectorizer.joblib
embeddings: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\smartphone_review_embeddings.npy
embedding_index: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\smartphone_review_embedding_index.csv
mock_c1: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\c1_mock_aspect_sentiment.csv
supported_minority_groups: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\mock_supported_minority_groups.csv

All required selector artifacts found.


## 2. Load and Validate Core Artifacts

The cleaned smartphone pilot is used as the master review-level dataset.

Each review must have:

- one stable `review_id`,
- one semantic embedding row,
- Function 1 usefulness inference support,
- zero or more C1-style aspect/sentiment records.


In [3]:
reviews_df = pd.read_parquet(
    PILOT_PATH
)


reviews_df["review_text"] = (
    reviews_df["title"]
    .fillna("")
    .astype(str)
    .str.strip()
    + " "
    + reviews_df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()


embedding_matrix = np.load(
    EMBEDDING_PATH
)


embedding_index_df = pd.read_csv(
    EMBEDDING_INDEX_PATH
)


c1_mock_df = pd.read_csv(
    C1_MOCK_PATH
)


supported_minority_df = pd.read_csv(
    SUPPORTED_MINORITY_PATH
)


loaded_model = joblib.load(
    MODEL_PATH
)


loaded_vectorizer = joblib.load(
    VECTORIZER_PATH
)


print(
    "Reviews:",
    len(reviews_df)
)

print(
    "Products:",
    reviews_df["parent_asin"].nunique()
)

print(
    "Embedding matrix:",
    embedding_matrix.shape
)

print(
    "Embedding index rows:",
    len(embedding_index_df)
)

print(
    "Mock C1 rows:",
    len(c1_mock_df)
)

print(
    "Supported-minority groups:",
    len(supported_minority_df)
)


Reviews: 896
Products: 10
Embedding matrix: (896, 384)
Embedding index rows: 896
Mock C1 rows: 1792
Supported-minority groups: 111


In [4]:
assert reviews_df[
    "review_id"
].is_unique

assert reviews_df[
    "review_id"
].notna().all()

assert (
    reviews_df[
        "review_text"
    ] != ""
).all()


assert (
    embedding_matrix.shape[0]
    == len(reviews_df)
)


assert (
    len(embedding_index_df)
    == len(reviews_df)
)


assert embedding_index_df[
    "review_id"
].is_unique


assert (
    set(
        reviews_df[
            "review_id"
        ]
    )
    ==
    set(
        embedding_index_df[
            "review_id"
        ]
    )
)


assert set(
    c1_mock_df[
        "review_id"
    ]
).issubset(
    set(
        reviews_df[
            "review_id"
        ]
    )
)


print(
    "Core selector artifact validation passed."
)


Core selector artifact validation passed.


## 3. Generate Function 1 Usefulness Scores

The saved Function 1 TF-IDF + Ridge model is applied to all candidate reviews.

The selector uses the **predicted usefulness score**.

`helpful_vote` is not used directly as a selector feature.

### Prototype Limitation

This step is intended to demonstrate integration with Function 1.

For final experimental evaluation, usefulness predictions should be generated using an evaluation protocol that avoids inappropriate train/evaluation overlap.


In [5]:
all_review_vectors = (
    loaded_vectorizer.transform(
        reviews_df[
            "review_text"
        ]
    )
)


all_usefulness_scores = (
    loaded_model.predict(
        all_review_vectors
    )
)


reviews_df[
    "predicted_usefulness"
] = (
    all_usefulness_scores
    .astype(float)
)


print(
    reviews_df[
        "predicted_usefulness"
    ].describe()
)


assert np.isfinite(
    reviews_df[
        "predicted_usefulness"
    ]
).all()


print(
    "Usefulness inference completed for all reviews."
)


count    896.000000
mean       0.586056
std        0.193404
min        0.176398
25%        0.443781
50%        0.557167
75%        0.690411
max        1.379281
Name: predicted_usefulness, dtype: float64
Usefulness inference completed for all reviews.


### Product-Relative Usefulness Normalization

Reviews are selected separately for each product.

A within-product min-max score is created so usefulness contributes on a consistent 0–1 scale.

The original predicted usefulness score is also retained.


In [6]:
def minmax_within_group(
    series
):
    minimum = series.min()
    maximum = series.max()

    if np.isclose(
        maximum,
        minimum,
    ):
        return pd.Series(
            np.zeros(
                len(series)
            ),
            index=series.index,
            dtype=float,
        )

    return (
        (series - minimum)
        / (maximum - minimum)
    )


reviews_df[
    "usefulness_norm"
] = (
    reviews_df
    .groupby(
        "parent_asin"
    )[
        "predicted_usefulness"
    ]
    .transform(
        minmax_within_group
    )
)


assert reviews_df[
    "usefulness_norm"
].between(
    0,
    1,
).all()


print(
    reviews_df[
        "usefulness_norm"
    ].describe()
)


print(
    "Within-product usefulness normalization passed."
)


count    896.000000
mean       0.382667
std        0.212636
min        0.000000
25%        0.223321
50%        0.352514
75%        0.506475
max        1.000000
Name: usefulness_norm, dtype: float64
Within-product usefulness normalization passed.


## 4. Attach Stable Embedding Rows

The selector uses an explicit `review_id → embedding_row` mapping instead of assuming that DataFrame order always matches the embedding matrix.


In [7]:
selector_df = reviews_df.merge(
    embedding_index_df[
        [
            "review_id",
            "embedding_row",
        ]
    ],
    on="review_id",
    how="left",
    validate="one_to_one",
)


assert selector_df[
    "embedding_row"
].notna().all()


selector_df[
    "embedding_row"
] = (
    selector_df[
        "embedding_row"
    ]
    .astype(int)
)


assert selector_df[
    "embedding_row"
].nunique() == len(
    selector_df
)


print(
    "Embedding-row mapping attached to",
    len(selector_df),
    "reviews."
)


Embedding-row mapping attached to 896 reviews.


## 5. Semantic Representativeness

For this prototype, semantic representativeness is estimated using the cosine similarity between each review embedding and the semantic centroid of all reviews for the same product.

Higher centroid similarity indicates that the review is semantically closer to the central review evidence for that product.

This is a preliminary operationalization and can later be compared with alternative representativeness measures.


In [8]:
representativeness_scores = pd.Series(
    index=selector_df.index,
    dtype=float,
)


for product_id, group in (
    selector_df.groupby(
        "parent_asin"
    )
):

    embedding_rows = (
        group[
            "embedding_row"
        ]
        .to_numpy(
            dtype=int
        )
    )


    product_embeddings = (
        embedding_matrix[
            embedding_rows
        ]
    )


    centroid = (
        product_embeddings
        .mean(
            axis=0,
            keepdims=True,
        )
    )


    centroid_norm = (
        np.linalg.norm(
            centroid
        )
    )


    if centroid_norm > 0:
        centroid = (
            centroid
            / centroid_norm
        )


    similarities = (
        cosine_similarity(
            product_embeddings,
            centroid,
        )
        .reshape(-1)
    )


    representativeness_scores.loc[
        group.index
    ] = similarities


selector_df[
    "representativeness"
] = (
    representativeness_scores
)


selector_df[
    "representativeness_norm"
] = (
    selector_df
    .groupby(
        "parent_asin"
    )[
        "representativeness"
    ]
    .transform(
        minmax_within_group
    )
)


assert np.isfinite(
    selector_df[
        "representativeness"
    ]
).all()


assert selector_df[
    "representativeness_norm"
].between(
    0,
    1,
).all()


display(
    selector_df[
        [
            "representativeness",
            "representativeness_norm",
        ]
    ].describe()
)


print(
    "Semantic representativeness calculation passed."
)


,representativeness,representativeness_norm
count,896.000000,896.000000
mean,0.612517,0.676332
std,0.162766,0.229068
min,0.034997,0.000000
25%,0.520296,0.549987
50%,0.649795,0.730549
75%,0.739843,0.853803
max,0.876348,1.000000


Semantic representativeness calculation passed.


## 6. Attach C1-Style Aspect/Sentiment Membership

A review can contain multiple aspect/sentiment records.

The current mock C1 records are aggregated into compact review-level fields:

- `aspect_categories`
- `aspect_sentiment_groups`

These remain synthetic integration-test inputs until the real C1 output becomes available.


In [9]:
c1_mock_df[
    "aspect_category"
] = (
    c1_mock_df[
        "aspect_category"
    ]
    .astype(str)
)


c1_mock_df[
    "sentiment"
] = (
    c1_mock_df[
        "sentiment"
    ]
    .astype(str)
)


c1_mock_df[
    "aspect_sentiment_key"
] = (
    c1_mock_df[
        "aspect_category"
    ]
    + "::"
    + c1_mock_df[
        "sentiment"
    ]
)


review_aspect_summary_df = (
    c1_mock_df
    .groupby(
        "review_id",
        as_index=False,
    )
    .agg(
        aspect_categories=(
            "aspect_category",
            lambda values:
                "|".join(
                    sorted(
                        set(values)
                    )
                ),
        ),
        aspect_sentiment_groups=(
            "aspect_sentiment_key",
            lambda values:
                "|".join(
                    sorted(
                        set(values)
                    )
                ),
        ),
    )
)


selector_df = selector_df.merge(
    review_aspect_summary_df,
    on="review_id",
    how="left",
    validate="one_to_one",
)


selector_df[
    "aspect_categories"
] = selector_df[
    "aspect_categories"
].fillna("")


selector_df[
    "aspect_sentiment_groups"
] = selector_df[
    "aspect_sentiment_groups"
].fillna("")


print(
    "Reviews with C1-style aspect records:",
    (
        selector_df[
            "aspect_categories"
        ] != ""
    ).sum()
)


display(
    selector_df[
        [
            "review_id",
            "parent_asin",
            "aspect_categories",
            "aspect_sentiment_groups",
        ]
    ].head()
)


Reviews with C1-style aspect records: 896


,review_id,parent_asin,aspect_categories,aspect_sentiment_groups
0,26c3cd248dc3d01d,B0B8PDLMMS,battery|camera,battery::positive|camera::neutral
1,cd97cd121c0fc4b7,B09HFW8GRB,camera|performance,camera::neutral|performance::negative
2,293639dc0ec21eae,B09JJCXC2M,display|performance,display::positive|performance::negative
3,5a4857744877f56c,B09JJCXC2M,display|price,display::neutral|price::negative
4,ca2d0d6e3ea776f8,B09PPWVQSB,design|price,design::neutral|price::positive


## 7. Attach Supported-Minority Membership

A review is marked as supporting a minority group when one of its aspect-sentiment records matches a group that passed the supported-minority qualification pipeline in Notebook 04.

Minority membership does not automatically make a review the highest-priority review.

The selector still balances usefulness, representativeness, coverage, redundancy, and budget feasibility.


In [10]:
supported_minority_df[
    "aspect_sentiment_key"
] = (
    supported_minority_df[
        "aspect_category"
    ].astype(str)
    + "::"
    + supported_minority_df[
        "sentiment"
    ].astype(str)
)


supported_keys_by_product = (
    supported_minority_df
    .groupby(
        "parent_asin"
    )[
        "aspect_sentiment_key"
    ]
    .apply(set)
    .to_dict()
)


def parse_pipe_set(
    value
):
    if pd.isna(
        value
    ):
        return set()

    return {
        item
        for item in str(
            value
        ).split("|")
        if item
    }


def get_supported_minority_keys(
    row
):
    product_keys = (
        supported_keys_by_product
        .get(
            row[
                "parent_asin"
            ],
            set(),
        )
    )


    review_keys = (
        parse_pipe_set(
            row[
                "aspect_sentiment_groups"
            ]
        )
    )


    matching_keys = sorted(
        review_keys
        & product_keys
    )


    return "|".join(
        matching_keys
    )


selector_df[
    "supported_minority_groups"
] = selector_df.apply(
    get_supported_minority_keys,
    axis=1,
)


selector_df[
    "supports_minority"
] = (
    selector_df[
        "supported_minority_groups"
    ]
    != ""
)


print(
    "Reviews supporting at least one "
    "mock supported-minority group:",
    selector_df[
        "supports_minority"
    ].sum()
)


display(
    selector_df[
        [
            "review_id",
            "parent_asin",
            "aspect_sentiment_groups",
            "supported_minority_groups",
            "supports_minority",
        ]
    ].head(10)
)


Reviews supporting at least one mock supported-minority group: 576


,review_id,parent_asin,aspect_sentiment_groups,supported_minority_groups,supports_minority
0,26c3cd248dc3d01d,B0B8PDLMMS,battery::positive|camera::neutral,camera::neutral,True
1,cd97cd121c0fc4b7,B09HFW8GRB,camera::neutral|performance::negative,camera::neutral|performance::negative,True
2,293639dc0ec21eae,B09JJCXC2M,display::positive|performance::negative,performance::negative,True
3,5a4857744877f56c,B09JJCXC2M,display::neutral|price::negative,display::neutral|price::negative,True
4,ca2d0d6e3ea776f8,B09PPWVQSB,design::neutral|price::positive,design::neutral,True
5,4b4e46ac390763fa,B0B8PDLMMS,battery::negative|design::positive,battery::negative,True
6,7ddd5e2ee3440476,B09JJCXC2M,battery::negative|camera::positive,battery::negative,True
7,198e0392108dbf2a,B09JJCXC2M,camera::negative|performance::positive,camera::negative,True
8,48b880bea79ad74a,B08XRDVWQ2,display::positive|performance::neutral,performance::neutral,True
9,4a09787cf2051d95,B0BL9TKSBL,display::positive|price::positive,,False


## 8. Build the Selector-Ready Review Table

The selector-ready table combines the main per-review inputs required by the constrained greedy algorithm.

### Real-data-derived fields

- `review_id`
- `parent_asin`
- `review_date`
- predicted usefulness
- normalized usefulness
- semantic embedding row
- semantic representativeness

### Mock C1-derived fields

- aspect categories
- aspect-sentiment groups
- supported-minority groups
- minority-support flag


In [11]:
selector_ready_df = selector_df[
    [
        "review_id",
        "parent_asin",
        "review_date",
        "predicted_usefulness",
        "usefulness_norm",
        "embedding_row",
        "representativeness",
        "representativeness_norm",
        "aspect_categories",
        "aspect_sentiment_groups",
        "supported_minority_groups",
        "supports_minority",
    ]
].copy()


assert (
    len(selector_ready_df)
    == len(reviews_df)
)


assert selector_ready_df[
    "review_id"
].is_unique


assert selector_ready_df[
    "embedding_row"
].is_unique


assert selector_ready_df[
    "usefulness_norm"
].between(
    0,
    1,
).all()


assert selector_ready_df[
    "representativeness_norm"
].between(
    0,
    1,
).all()


print(
    "Selector-ready rows:",
    len(selector_ready_df)
)


print(
    "Selector-ready products:",
    selector_ready_df[
        "parent_asin"
    ].nunique()
)


print(
    "\nMissing values:"
)


print(
    selector_ready_df
    .isna()
    .sum()
)


print(
    "\nSelector-ready table validation passed."
)


display(
    selector_ready_df.head(10)
)


Selector-ready rows: 896
Selector-ready products: 10

Missing values:
review_id                    0
parent_asin                  0
review_date                  0
predicted_usefulness         0
usefulness_norm              0
embedding_row                0
representativeness           0
representativeness_norm      0
aspect_categories            0
aspect_sentiment_groups      0
supported_minority_groups    0
supports_minority            0
dtype: int64

Selector-ready table validation passed.


,review_id,parent_asin,review_date,predicted_usefulness,usefulness_norm,embedding_row,representativeness,representativeness_norm,aspect_categories,aspect_sentiment_groups,supported_minority_groups,supports_minority
0,26c3cd248dc3d01d,B0B8PDLMMS,2021-09-08 23:51:02.923000+00:00,0.448435,0.112543,0,0.765456,0.897396,battery|camera,battery::positive|camera::neutral,camera::neutral,True
1,cd97cd121c0fc4b7,B09HFW8GRB,2021-06-05 21:25:08.584000+00:00,0.341173,0.138272,1,0.159073,0.000000,camera|performance,camera::neutral|performance::negative,camera::neutral|performance::negative,True
2,293639dc0ec21eae,B09JJCXC2M,2021-09-19 13:25:13.275000+00:00,0.606561,0.421915,2,0.754523,0.904733,display|performance,display::positive|performance::negative,performance::negative,True
3,5a4857744877f56c,B09JJCXC2M,2020-05-19 20:45:45.585000+00:00,0.535712,0.341301,3,0.676295,0.794841,display|price,display::neutral|price::negative,display::neutral|price::negative,True
4,ca2d0d6e3ea776f8,B09PPWVQSB,2022-04-18 21:49:41.193000+00:00,0.587383,0.352430,4,0.708842,0.871760,design|price,design::neutral|price::positive,design::neutral,True
5,4b4e46ac390763fa,B0B8PDLMMS,2022-06-09 06:52:21.382000+00:00,0.587237,0.244875,5,0.745228,0.866833,battery|design,battery::negative|design::positive,battery::negative,True
6,7ddd5e2ee3440476,B09JJCXC2M,2023-03-01 02:49:21.830000+00:00,1.114625,1.000000,6,0.688781,0.812380,battery|camera,battery::negative|camera::positive,battery::negative,True
7,198e0392108dbf2a,B09JJCXC2M,2022-11-15 18:45:05.184000+00:00,0.637570,0.457197,7,0.804102,0.974380,camera|performance,camera::negative|performance::positive,camera::negative,True
8,48b880bea79ad74a,B08XRDVWQ2,2019-10-18 01:53:08.843000+00:00,0.685741,0.515032,8,0.859236,1.000000,display|performance,display::positive|performance::neutral,performance::neutral,True
9,4a09787cf2051d95,B0BL9TKSBL,2021-12-01 17:35:55.832000+00:00,0.932228,0.929243,9,0.667211,0.698644,display|price,display::positive|price::positive,,False


In [12]:
SELECTOR_READY_PATH = (
    SELECTOR_RESULTS_DIR
    / "selector_ready_reviews.csv"
)


SELECTOR_PREP_SUMMARY_PATH = (
    SELECTOR_RESULTS_DIR
    / "selector_preparation_summary.json"
)


selector_ready_df.to_csv(
    SELECTOR_READY_PATH,
    index=False,
)


selector_preparation_summary = {
    "reviews":
        int(
            len(
                selector_ready_df
            )
        ),

    "products":
        int(
            selector_ready_df[
                "parent_asin"
            ].nunique()
        ),

    "embedding_dimension":
        int(
            embedding_matrix.shape[1]
        ),

    "reviews_supporting_mock_minority":
        int(
            selector_ready_df[
                "supports_minority"
            ].sum()
        ),

    "mock_c1_warning":
        (
            "Aspect/sentiment and minority fields "
            "are synthetic integration-test placeholders."
        ),
}


with open(
    SELECTOR_PREP_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        selector_preparation_summary,
        f,
        indent=2,
    )


print(
    "Selector-ready table saved:",
    SELECTOR_READY_PATH
)


print(
    "Selector preparation summary saved:",
    SELECTOR_PREP_SUMMARY_PATH
)


Selector-ready table saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_ready_reviews.csv
Selector preparation summary saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_preparation_summary.json


## 9. Provisional Selector Configuration

The first prototype uses a fixed budget of **k = 5 reviews per product**.

The candidate score combines:

- normalized usefulness,
- normalized semantic representativeness,
- new aspect coverage,
- new supported-minority coverage,
- semantic redundancy penalty.

These settings are provisional and must later be examined through sensitivity and ablation analysis.


In [13]:
SELECT_K = 5


WEIGHT_USEFULNESS = 0.30

WEIGHT_REPRESENTATIVENESS = 0.25

WEIGHT_COVERAGE = 0.20

WEIGHT_MINORITY = 0.25

WEIGHT_REDUNDANCY = 0.20


MINORITY_PRIORITY_GROUPS = 2

MINORITY_RESERVED_SLOTS = 2


print(
    "Prototype review budget:",
    SELECT_K
)


print(
    "Weights:",
    {
        "usefulness":
            WEIGHT_USEFULNESS,

        "representativeness":
            WEIGHT_REPRESENTATIVENESS,

        "coverage":
            WEIGHT_COVERAGE,

        "minority":
            WEIGHT_MINORITY,

        "redundancy_penalty":
            WEIGHT_REDUNDANCY,
    }
)


print(
    "Priority minority groups:",
    MINORITY_PRIORITY_GROUPS
)


print(
    "Reserved minority slots:",
    MINORITY_RESERVED_SLOTS
)


print(
    "\nThese are provisional prototype settings."
)


Prototype review budget: 5
Weights: {'usefulness': 0.3, 'representativeness': 0.25, 'coverage': 0.2, 'minority': 0.25, 'redundancy_penalty': 0.2}
Priority minority groups: 2
Reserved minority slots: 2

These are provisional prototype settings.


## 10. Determine Budget-Feasible Priority Minority Groups

A product can contain more supported-minority groups than can be preserved within a small fixed review budget.

The prototype therefore identifies a small budget-feasible priority subset.

Priority is based on:

1. lower prevalence,
2. higher support count,
3. higher average confidence.

This implements the principle:

**minority preservation ≠ minority amplification**


In [14]:
priority_minority_by_product = {}


for product_id, group in (
    supported_minority_df
    .groupby(
        "parent_asin"
    )
):

    sorted_group = (
        group
        .sort_values(
            [
                "prevalence",
                "support_count",
                "avg_confidence",
            ],
            ascending=[
                True,
                False,
                False,
            ],
        )
        .drop_duplicates(
            subset=[
                "aspect_sentiment_key"
            ]
        )
    )


    priority_keys = (
        sorted_group[
            "aspect_sentiment_key"
        ]
        .head(
            MINORITY_PRIORITY_GROUPS
        )
        .tolist()
    )


    priority_minority_by_product[
        product_id
    ] = set(
        priority_keys
    )


priority_preview = []


for product_id, keys in (
    priority_minority_by_product.items()
):

    priority_preview.append(
        {
            "parent_asin":
                product_id,

            "priority_minority_groups":
                "|".join(
                    sorted(keys)
                ),

            "num_priority_groups":
                len(keys),
        }
    )


priority_preview_df = (
    pd.DataFrame(
        priority_preview
    )
)


display(
    priority_preview_df
)


,parent_asin,priority_minority_groups,num_priority_groups
0,B07GYY6VTD,camera::negative|performance::neutral,2
1,B07SDQ483V,battery::neutral|camera::negative,2
2,B08PP9FM9N,battery::negative|camera::negative,2
3,B08XRDVWQ2,battery::negative|battery::neutral,2
4,B09HFW8GRB,battery::negative|camera::negative,2
5,B09JJCXC2M,battery::neutral|performance::neutral,2
6,B09PPWVQSB,design::neutral|price::neutral,2
7,B0B8PDLMMS,camera::negative|performance::neutral,2
8,B0BL9TKSBL,camera::neutral|display::neutral,2
9,B0BMWTL2S6,camera::negative|display::negative,2


## 11. Candidate Scoring Functions

### Semantic Redundancy

Redundancy is the maximum cosine similarity between a candidate review and any review already selected.

Higher similarity indicates stronger repetition.

### Coverage Gain

Coverage gain measures the fraction of product aspects newly introduced by the candidate.

### Minority Gain

Minority gain measures newly covered priority supported-minority groups.

### Candidate Score

The provisional score is:

`usefulness + representativeness + coverage gain + minority gain - redundancy`


In [15]:
def candidate_redundancy(
    candidate_embedding_row,
    selected_embedding_rows,
    embeddings,
):

    if len(
        selected_embedding_rows
    ) == 0:
        return 0.0


    candidate_vector = (
        embeddings[
            int(
                candidate_embedding_row
            )
        ]
        .reshape(
            1,
            -1,
        )
    )


    selected_vectors = (
        embeddings[
            np.asarray(
                selected_embedding_rows,
                dtype=int,
            )
        ]
    )


    similarities = (
        cosine_similarity(
            candidate_vector,
            selected_vectors,
        )[0]
    )


    redundancy = float(
        np.max(
            similarities
        )
    )


    return max(
        0.0,
        redundancy,
    )


In [16]:
def candidate_score(
    row,
    selected_embedding_rows,
    covered_aspects,
    total_product_aspects,
    covered_priority_groups,
    priority_groups,
):

    candidate_aspects = (
        parse_pipe_set(
            row[
                "aspect_categories"
            ]
        )
    )


    candidate_minority_groups = (
        parse_pipe_set(
            row[
                "supported_minority_groups"
            ]
        )
    )


    new_aspects = (
        candidate_aspects
        - covered_aspects
    )


    uncovered_priority_groups = (
        priority_groups
        - covered_priority_groups
    )


    new_priority_groups = (
        candidate_minority_groups
        & uncovered_priority_groups
    )


    coverage_gain = (
        len(
            new_aspects
        )
        /
        max(
            len(
                total_product_aspects
            ),
            1,
        )
    )


    if priority_groups:

        minority_gain = (
            len(
                new_priority_groups
            )
            /
            len(
                priority_groups
            )
        )

    else:
        minority_gain = 0.0


    redundancy = (
        candidate_redundancy(
            row[
                "embedding_row"
            ],
            selected_embedding_rows,
            embedding_matrix,
        )
    )


    score = (
        WEIGHT_USEFULNESS
        * float(
            row[
                "usefulness_norm"
            ]
        )

        + WEIGHT_REPRESENTATIVENESS
        * float(
            row[
                "representativeness_norm"
            ]
        )

        + WEIGHT_COVERAGE
        * coverage_gain

        + WEIGHT_MINORITY
        * minority_gain

        - WEIGHT_REDUNDANCY
        * redundancy
    )


    return {
        "selection_score":
            float(score),

        "coverage_gain":
            float(
                coverage_gain
            ),

        "minority_gain":
            float(
                minority_gain
            ),

        "redundancy":
            float(
                redundancy
            ),

        "new_aspects":
            "|".join(
                sorted(
                    new_aspects
                )
            ),

        "new_priority_minority_groups":
            "|".join(
                sorted(
                    new_priority_groups
                )
            ),
    }


## 12. Two-Phase Constrained Greedy Selector

The selection procedure contains two phases.

### Phase A – Minority-Preservation Reserve

A small number of slots are reserved to cover priority supported-minority groups where feasible.

### Phase B – General Greedy Selection

The remaining slots are filled using the complete candidate score.

This is a first prototype heuristic and not yet the final optimized formulation.


In [17]:
def select_product_evidence(
    product_df,
    budget,
):

    product_df = (
        product_df
        .copy()
        .reset_index(
            drop=True
        )
    )


    product_id = (
        product_df.loc[
            0,
            "parent_asin",
        ]
    )


    priority_groups = (
        priority_minority_by_product
        .get(
            product_id,
            set(),
        )
    )


    total_product_aspects = set()


    for value in (
        product_df[
            "aspect_categories"
        ]
    ):

        total_product_aspects |= (
            parse_pipe_set(
                value
            )
        )


    selected_indices = []

    selected_embedding_rows = []

    covered_aspects = set()

    covered_priority_groups = set()

    selection_records = []


    def evaluate_candidate(
        idx
    ):

        row = (
            product_df.loc[
                idx
            ]
        )


        diagnostics = (
            candidate_score(
                row=row,

                selected_embedding_rows=
                    selected_embedding_rows,

                covered_aspects=
                    covered_aspects,

                total_product_aspects=
                    total_product_aspects,

                covered_priority_groups=
                    covered_priority_groups,

                priority_groups=
                    priority_groups,
            )
        )


        return {
            "idx":
                idx,

            "review_id":
                row[
                    "review_id"
                ],

            "tie_usefulness":
                float(
                    row[
                        "usefulness_norm"
                    ]
                ),

            **diagnostics,
        }


    # =====================================================
    # Phase A
    # Minority-preservation reserve
    # =====================================================

    reserve_limit = min(
        MINORITY_RESERVED_SLOTS,
        budget,
    )


    while (
        len(
            selected_indices
        )
        < reserve_limit
    ):

        uncovered_priority = (
            priority_groups
            - covered_priority_groups
        )


        if not uncovered_priority:
            break


        eligible = []


        for idx, row in (
            product_df.iterrows()
        ):

            if idx in selected_indices:
                continue


            candidate_groups = (
                parse_pipe_set(
                    row[
                        "supported_minority_groups"
                    ]
                )
            )


            if (
                candidate_groups
                & uncovered_priority
            ):

                eligible.append(
                    evaluate_candidate(
                        idx
                    )
                )


        if not eligible:
            break


        best = max(
            eligible,
            key=lambda item: (
                item[
                    "minority_gain"
                ],
                item[
                    "selection_score"
                ],
                item[
                    "coverage_gain"
                ],
                item[
                    "tie_usefulness"
                ],
            ),
        )


        best_idx = (
            best[
                "idx"
            ]
        )


        best_row = (
            product_df.loc[
                best_idx
            ]
        )


        selected_indices.append(
            best_idx
        )


        selected_embedding_rows.append(
            int(
                best_row[
                    "embedding_row"
                ]
            )
        )


        covered_aspects |= (
            parse_pipe_set(
                best_row[
                    "aspect_categories"
                ]
            )
        )


        covered_priority_groups |= (
            parse_pipe_set(
                best_row[
                    "supported_minority_groups"
                ]
            )
            & priority_groups
        )


        selection_records.append(
            {
                "review_id":
                    best_row[
                        "review_id"
                    ],

                "selection_phase":
                    "minority_reserve",

                **{
                    key: value
                    for key, value
                    in best.items()
                    if key not in {
                        "idx",
                        "review_id",
                        "tie_usefulness",
                    }
                },
            }
        )


    # =====================================================
    # Phase B
    # General greedy selection
    # =====================================================

    target_count = min(
        budget,
        len(
            product_df
        ),
    )


    while (
        len(
            selected_indices
        )
        < target_count
    ):

        candidates = []


        for idx, row in (
            product_df.iterrows()
        ):

            if idx in selected_indices:
                continue


            candidates.append(
                evaluate_candidate(
                    idx
                )
            )


        if not candidates:
            break


        best = max(
            candidates,
            key=lambda item: (
                item[
                    "selection_score"
                ],
                item[
                    "minority_gain"
                ],
                item[
                    "coverage_gain"
                ],
                item[
                    "tie_usefulness"
                ],
            ),
        )


        best_idx = (
            best[
                "idx"
            ]
        )


        best_row = (
            product_df.loc[
                best_idx
            ]
        )


        selected_indices.append(
            best_idx
        )


        selected_embedding_rows.append(
            int(
                best_row[
                    "embedding_row"
                ]
            )
        )


        covered_aspects |= (
            parse_pipe_set(
                best_row[
                    "aspect_categories"
                ]
            )
        )


        covered_priority_groups |= (
            parse_pipe_set(
                best_row[
                    "supported_minority_groups"
                ]
            )
            & priority_groups
        )


        selection_records.append(
            {
                "review_id":
                    best_row[
                        "review_id"
                    ],

                "selection_phase":
                    "general_greedy",

                **{
                    key: value
                    for key, value
                    in best.items()
                    if key not in {
                        "idx",
                        "review_id",
                        "tie_usefulness",
                    }
                },
            }
        )


    selected_df = (
        product_df.loc[
            selected_indices
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    diagnostics_df = (
        pd.DataFrame(
            selection_records
        )
    )


    selected_df = (
        selected_df.merge(
            diagnostics_df,
            on="review_id",
            how="left",
            validate="one_to_one",
        )
    )


    selected_df[
        "selection_rank"
    ] = np.arange(
        1,
        len(
            selected_df
        ) + 1,
    )


    selected_df[
        "priority_groups_targeted"
    ] = "|".join(
        sorted(
            priority_groups
        )
    )


    selected_df[
        "priority_groups_covered_final"
    ] = "|".join(
        sorted(
            covered_priority_groups
        )
    )


    return selected_df


## 13. Run the Fixed-Budget Selector

The first prototype selects up to **five reviews per product**.


In [18]:
selected_product_frames = []


for product_id, group in (
    selector_ready_df
    .groupby(
        "parent_asin"
    )
):

    selected_product = (
        select_product_evidence(
            product_df=group,
            budget=SELECT_K,
        )
    )


    selected_product_frames.append(
        selected_product
    )


selected_evidence_df = (
    pd.concat(
        selected_product_frames,
        ignore_index=True,
    )
)


print(
    "Selected evidence rows:",
    len(
        selected_evidence_df
    )
)


print(
    "Products represented:",
    selected_evidence_df[
        "parent_asin"
    ].nunique()
)


selected_counts_df = (
    selected_evidence_df[
        "parent_asin"
    ]
    .value_counts()
    .rename_axis(
        "parent_asin"
    )
    .reset_index(
        name="selected_count"
    )
)


display(
    selected_counts_df
)


Selected evidence rows: 50
Products represented: 10


,parent_asin,selected_count
0,B07GYY6VTD,5
1,B07SDQ483V,5
2,B08PP9FM9N,5
3,B08XRDVWQ2,5
4,B09HFW8GRB,5
5,B09JJCXC2M,5
6,B09PPWVQSB,5
7,B0B8PDLMMS,5
8,B0BL9TKSBL,5
9,B0BMWTL2S6,5


In [19]:
assert selected_evidence_df[
    "review_id"
].is_unique


assert set(
    selected_evidence_df[
        "review_id"
    ]
).issubset(
    set(
        selector_ready_df[
            "review_id"
        ]
    )
)


for product_id, group in (
    selector_ready_df
    .groupby(
        "parent_asin"
    )
):

    expected_count = min(
        SELECT_K,
        len(group),
    )


    actual_count = len(
        selected_evidence_df[
            selected_evidence_df[
                "parent_asin"
            ] == product_id
        ]
    )


    assert (
        actual_count
        == expected_count
    )


assert np.isfinite(
    selected_evidence_df[
        "selection_score"
    ]
).all()


assert selected_evidence_df[
    "selection_phase"
].isin(
    [
        "minority_reserve",
        "general_greedy",
    ]
).all()


print(
    "Fixed-budget selector validation passed."
)


Fixed-budget selector validation passed.


## 14. Inspect Selected Evidence

The following table shows the score components that contributed to each selector decision.


In [20]:
selection_display_columns = [
    "parent_asin",
    "selection_rank",
    "review_id",
    "selection_phase",
    "usefulness_norm",
    "representativeness_norm",
    "coverage_gain",
    "minority_gain",
    "redundancy",
    "selection_score",
    "aspect_categories",
    "supported_minority_groups",
]


display(
    selected_evidence_df[
        selection_display_columns
    ]
    .sort_values(
        [
            "parent_asin",
            "selection_rank",
        ]
    )
    .head(30)
)


,parent_asin,selection_rank,review_id,selection_phase,usefulness_norm,representativeness_norm,coverage_gain,minority_gain,redundancy,selection_score,aspect_categories,supported_minority_groups
0,B07GYY6VTD,1,082b67f5c74a45b8,minority_reserve,0.296681,1.000000,0.333333,0.5,0.000000,0.530671,camera|performance,camera::negative
1,B07GYY6VTD,2,866978504b0f427f,minority_reserve,0.334609,0.718703,0.166667,0.5,0.555722,0.327248,display|performance,display::neutral|performance::neutral
2,B07GYY6VTD,3,6f8a74cde06a1711,general_greedy,1.000000,0.791508,0.166667,0.0,0.508111,0.429588,display|price,display::neutral|price::negative
3,B07GYY6VTD,4,b8fc62a6d1a0d912,general_greedy,0.659542,0.930690,0.333333,0.0,0.673888,0.362424,battery|design,
4,B07GYY6VTD,5,b068d451b1f068c1,general_greedy,0.900361,0.758019,0.000000,0.0,0.599541,0.339705,camera|performance,camera::neutral
5,B07SDQ483V,1,8ea0f6f697d7447e,minority_reserve,1.000000,0.664981,0.333333,0.5,0.000000,0.657912,camera|performance,camera::negative
6,B07SDQ483V,2,03fe2e44ba4656ed,minority_reserve,0.210692,1.000000,0.333333,0.5,0.548329,0.395208,battery|design,battery::neutral|design::neutral
7,B07SDQ483V,3,745c41e302104325,general_greedy,0.811519,0.889870,0.000000,0.0,0.697052,0.326513,battery|design,design::neutral
8,B07SDQ483V,4,72b8dd78d5aeed90,general_greedy,0.606043,0.752773,0.333333,0.0,0.649297,0.306813,display|price,
9,B07SDQ483V,5,092068c7d5262747,general_greedy,0.629610,0.892387,0.000000,0.0,0.696442,0.272691,battery|camera,


In [21]:
selected_preview_df = (
    selected_evidence_df.merge(
        reviews_df[
            [
                "review_id",
                "review_text",
            ]
        ],
        on="review_id",
        how="left",
        validate="one_to_one",
    )
)


selected_preview_df[
    "review_text_preview"
] = (
    selected_preview_df[
        "review_text"
    ]
    .str.slice(
        0,
        250,
    )
)


sample_product_id = (
    selected_preview_df[
        "parent_asin"
    ]
    .iloc[0]
)


display(
    selected_preview_df[
        selected_preview_df[
            "parent_asin"
        ] == sample_product_id
    ][
        [
            "selection_rank",
            "review_id",
            "selection_phase",
            "selection_score",
            "review_text_preview",
        ]
    ]
)


,selection_rank,review_id,selection_phase,selection_score,review_text_preview
0,1,082b67f5c74a45b8,minority_reserve,0.530671,Best phone I've had yet I love this phone so m...
1,2,866978504b0f427f,minority_reserve,0.327248,BREAKS EASILY This phone breaks so easily! I h...
2,3,6f8a74cde06a1711,general_greedy,0.429588,ITS A LENOVO PHONE! + 6 MONTHS UPDATE What a d...
3,4,b8fc62a6d1a0d912,general_greedy,0.362424,Excellent performance great price I lost other...
4,5,b068d451b1f068c1,general_greedy,0.339705,So far I am not thrilled with the performance ...


## 15. Prototype Selection Evaluation

The following diagnostics are calculated.

### Usefulness
Mean normalized predicted usefulness.

### Representativeness
Mean normalized semantic representativeness.

### Aspect Coverage
Fraction of product aspects represented by the selected set.

### Supported Minority Recall
Fraction of all qualified supported-minority groups represented by the selected set.

### Priority Minority Coverage
Coverage of the small budget-feasible priority minority subset.

### Redundancy
Mean pairwise cosine similarity between selected reviews. Lower indicates less semantic repetition.

### Opinion Fidelity
Jensen-Shannon divergence between the full product aspect-sentiment distribution and the selected-set distribution.

Lower JSD indicates better distributional fidelity.

Because the current C1 information is synthetic, aspect, minority, and fidelity metrics are integration-test diagnostics only.


In [22]:
def mean_pairwise_similarity(
    embedding_rows
):

    embedding_rows = list(
        embedding_rows
    )


    if len(
        embedding_rows
    ) < 2:
        return 0.0


    vectors = (
        embedding_matrix[
            np.asarray(
                embedding_rows,
                dtype=int,
            )
        ]
    )


    similarity_matrix = (
        cosine_similarity(
            vectors
        )
    )


    upper_triangle = (
        np.triu_indices(
            len(
                embedding_rows
            ),
            k=1,
        )
    )


    return float(
        np.mean(
            similarity_matrix[
                upper_triangle
            ]
        )
    )


In [23]:
def product_opinion_jsd(
    product_id,
    selected_review_ids,
):

    full_product_review_ids = set(
        selector_ready_df[
            selector_ready_df[
                "parent_asin"
            ] == product_id
        ][
            "review_id"
        ]
    )


    full_c1 = (
        c1_mock_df[
            c1_mock_df[
                "review_id"
            ].isin(
                full_product_review_ids
            )
        ]
    )


    selected_c1 = (
        c1_mock_df[
            c1_mock_df[
                "review_id"
            ].isin(
                set(
                    selected_review_ids
                )
            )
        ]
    )


    categories = sorted(
        set(
            full_c1[
                "aspect_sentiment_key"
            ]
        )
        |
        set(
            selected_c1[
                "aspect_sentiment_key"
            ]
        )
    )


    if not categories:
        return np.nan


    full_counts = (
        full_c1[
            "aspect_sentiment_key"
        ]
        .value_counts()
    )


    selected_counts = (
        selected_c1[
            "aspect_sentiment_key"
        ]
        .value_counts()
    )


    p = np.array(
        [
            full_counts.get(
                key,
                0,
            )
            for key in categories
        ],
        dtype=float,
    )


    q = np.array(
        [
            selected_counts.get(
                key,
                0,
            )
            for key in categories
        ],
        dtype=float,
    )


    if (
        p.sum() <= 0
        or q.sum() <= 0
    ):
        return np.nan


    p = p / p.sum()

    q = q / q.sum()


    js_distance = (
        jensenshannon(
            p,
            q,
            base=2.0,
        )
    )


    return float(
        js_distance ** 2
    )


In [24]:
def summarize_selection(
    selection_df,
    selector_name,
):

    records = []


    for product_id, full_group in (
        selector_ready_df
        .groupby(
            "parent_asin"
        )
    ):

        selected_group = (
            selection_df[
                selection_df[
                    "parent_asin"
                ] == product_id
            ]
        )


        all_aspects = set()


        for value in (
            full_group[
                "aspect_categories"
            ]
        ):

            all_aspects |= (
                parse_pipe_set(
                    value
                )
            )


        selected_aspects = set()


        for value in (
            selected_group[
                "aspect_categories"
            ]
        ):

            selected_aspects |= (
                parse_pipe_set(
                    value
                )
            )


        all_supported_groups = (
            supported_keys_by_product
            .get(
                product_id,
                set(),
            )
        )


        selected_supported_groups = set()


        for value in (
            selected_group[
                "supported_minority_groups"
            ]
        ):

            selected_supported_groups |= (
                parse_pipe_set(
                    value
                )
            )


        priority_groups = (
            priority_minority_by_product
            .get(
                product_id,
                set(),
            )
        )


        covered_priority_groups = (
            selected_supported_groups
            & priority_groups
        )


        aspect_coverage = (
            len(
                selected_aspects
            )
            /
            max(
                len(
                    all_aspects
                ),
                1,
            )
        )


        if all_supported_groups:

            supported_minority_recall = (
                len(
                    selected_supported_groups
                    & all_supported_groups
                )
                /
                len(
                    all_supported_groups
                )
            )

        else:
            supported_minority_recall = np.nan


        if priority_groups:

            priority_minority_coverage = (
                len(
                    covered_priority_groups
                )
                /
                len(
                    priority_groups
                )
            )

        else:
            priority_minority_coverage = np.nan


        pairwise_similarity = (
            mean_pairwise_similarity(
                selected_group[
                    "embedding_row"
                ].tolist()
            )
        )


        js_divergence = (
            product_opinion_jsd(
                product_id,
                selected_group[
                    "review_id"
                ].tolist(),
            )
        )


        records.append(
            {
                "selector":
                    selector_name,

                "parent_asin":
                    product_id,

                "selected_count":
                    len(
                        selected_group
                    ),

                "mean_usefulness_norm":
                    float(
                        selected_group[
                            "usefulness_norm"
                        ].mean()
                    ),

                "mean_representativeness_norm":
                    float(
                        selected_group[
                            "representativeness_norm"
                        ].mean()
                    ),

                "aspect_coverage":
                    float(
                        aspect_coverage
                    ),

                "supported_minority_recall":
                    float(
                        supported_minority_recall
                    )
                    if not np.isnan(
                        supported_minority_recall
                    )
                    else np.nan,

                "priority_minority_coverage":
                    float(
                        priority_minority_coverage
                    )
                    if not np.isnan(
                        priority_minority_coverage
                    )
                    else np.nan,

                "mean_pairwise_similarity":
                    float(
                        pairwise_similarity
                    ),

                "opinion_js_divergence":
                    float(
                        js_divergence
                    )
                    if not np.isnan(
                        js_divergence
                    )
                    else np.nan,
            }
        )


    return pd.DataFrame(
        records
    )


In [25]:
proposed_product_metrics_df = (
    summarize_selection(
        selected_evidence_df,
        "ProposedPrototype",
    )
)


display(
    proposed_product_metrics_df
)


,selector,parent_asin,selected_count,mean_usefulness_norm,mean_representativeness_norm,aspect_coverage,supported_minority_recall,priority_minority_coverage,mean_pairwise_similarity,opinion_js_divergence
0,ProposedPrototype,B07GYY6VTD,5,0.638239,0.839784,1.000000,0.454545,1.0,0.556951,0.387555
1,ProposedPrototype,B07SDQ483V,5,0.651573,0.840002,1.000000,0.272727,1.0,0.585264,0.269152
2,ProposedPrototype,B08PP9FM9N,5,0.677433,0.824097,1.000000,0.444444,1.0,0.558804,0.393369
3,ProposedPrototype,B08XRDVWQ2,5,0.909621,0.780334,1.000000,0.333333,1.0,0.518553,0.366642
4,ProposedPrototype,B09HFW8GRB,5,0.709492,0.816158,0.666667,0.416667,1.0,0.499674,0.376623
5,ProposedPrototype,B09JJCXC2M,5,0.817242,0.709770,0.833333,0.500000,1.0,0.431964,0.405713
6,ProposedPrototype,B09PPWVQSB,5,0.792421,0.834772,1.000000,0.384615,1.0,0.517966,0.289642
7,ProposedPrototype,B0B8PDLMMS,5,0.709414,0.596574,1.000000,0.444444,1.0,0.306404,0.284042
8,ProposedPrototype,B0BL9TKSBL,5,0.745937,0.693983,0.833333,0.416667,1.0,0.442127,0.369518
9,ProposedPrototype,B0BMWTL2S6,5,0.775876,0.730492,1.000000,0.500000,1.0,0.508401,0.225451


## 16. Predicted-Usefulness Top-K Baseline

The initial comparison baseline selects the `k` reviews with the highest predicted usefulness for each product.

This allows us to inspect whether the constrained prototype trades some usefulness for:

- broader aspect coverage,
- minority preservation,
- lower redundancy,
- improved opinion fidelity.

Additional baselines remain later work.


In [26]:
baseline_frames = []


for product_id, group in (
    selector_ready_df
    .groupby(
        "parent_asin"
    )
):

    baseline_group = (
        group
        .sort_values(
            [
                "usefulness_norm",
                "predicted_usefulness",
                "review_id",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .head(
            SELECT_K
        )
        .copy()
    )


    baseline_frames.append(
        baseline_group
    )


usefulness_topk_df = (
    pd.concat(
        baseline_frames,
        ignore_index=True,
    )
)


print(
    "Usefulness Top-K rows:",
    len(
        usefulness_topk_df
    )
)


assert usefulness_topk_df[
    "review_id"
].is_unique


Usefulness Top-K rows: 50


In [27]:
baseline_product_metrics_df = (
    summarize_selection(
        usefulness_topk_df,
        "UsefulnessTopK",
    )
)


selector_comparison_product_df = (
    pd.concat(
        [
            proposed_product_metrics_df,
            baseline_product_metrics_df,
        ],
        ignore_index=True,
    )
)


display(
    selector_comparison_product_df.head(
        20
    )
)


,selector,parent_asin,selected_count,mean_usefulness_norm,mean_representativeness_norm,aspect_coverage,supported_minority_recall,priority_minority_coverage,mean_pairwise_similarity,opinion_js_divergence
0,ProposedPrototype,B07GYY6VTD,5,0.638239,0.839784,1.000000,0.454545,1.0,0.556951,0.387555
1,ProposedPrototype,B07SDQ483V,5,0.651573,0.840002,1.000000,0.272727,1.0,0.585264,0.269152
2,ProposedPrototype,B08PP9FM9N,5,0.677433,0.824097,1.000000,0.444444,1.0,0.558804,0.393369
3,ProposedPrototype,B08XRDVWQ2,5,0.909621,0.780334,1.000000,0.333333,1.0,0.518553,0.366642
4,ProposedPrototype,B09HFW8GRB,5,0.709492,0.816158,0.666667,0.416667,1.0,0.499674,0.376623
5,ProposedPrototype,B09JJCXC2M,5,0.817242,0.709770,0.833333,0.500000,1.0,0.431964,0.405713
6,ProposedPrototype,B09PPWVQSB,5,0.792421,0.834772,1.000000,0.384615,1.0,0.517966,0.289642
7,ProposedPrototype,B0B8PDLMMS,5,0.709414,0.596574,1.000000,0.444444,1.0,0.306404,0.284042
8,ProposedPrototype,B0BL9TKSBL,5,0.745937,0.693983,0.833333,0.416667,1.0,0.442127,0.369518
9,ProposedPrototype,B0BMWTL2S6,5,0.775876,0.730492,1.000000,0.500000,1.0,0.508401,0.225451


In [28]:
selector_comparison_overall_df = (
    selector_comparison_product_df
    .groupby(
        "selector",
        as_index=False,
    )
    .agg(
        mean_usefulness_norm=(
            "mean_usefulness_norm",
            "mean",
        ),

        mean_representativeness_norm=(
            "mean_representativeness_norm",
            "mean",
        ),

        mean_aspect_coverage=(
            "aspect_coverage",
            "mean",
        ),

        mean_supported_minority_recall=(
            "supported_minority_recall",
            "mean",
        ),

        mean_priority_minority_coverage=(
            "priority_minority_coverage",
            "mean",
        ),

        mean_pairwise_similarity=(
            "mean_pairwise_similarity",
            "mean",
        ),

        mean_opinion_js_divergence=(
            "opinion_js_divergence",
            "mean",
        ),
    )
)


display(
    selector_comparison_overall_df
)


,selector,mean_usefulness_norm,mean_representativeness_norm,mean_aspect_coverage,mean_supported_minority_recall,mean_priority_minority_coverage,mean_pairwise_similarity,mean_opinion_js_divergence
0,ProposedPrototype,0.742725,0.766597,0.933333,0.416744,1.0,0.492611,0.336771
1,UsefulnessTopK,0.859078,0.711964,0.816667,0.278294,0.1,0.472905,0.316286


### Prototype Comparison Interpretation

For the comparison metrics:

- higher usefulness is better,
- higher representativeness is better,
- higher aspect coverage is better,
- higher supported-minority recall is better,
- higher priority-minority coverage is better,
- lower pairwise similarity indicates less redundant evidence,
- lower Jensen-Shannon divergence indicates better opinion-distribution fidelity.

Because the current C1 aspect/sentiment information is synthetic, coverage, minority, and opinion-fidelity results are integration-test diagnostics only.


## 17. Save Selector Outputs and Configuration

Compact selector artifacts are saved for reproducibility and PP1 evidence.


In [29]:
SELECTED_EVIDENCE_PATH = (
    SELECTOR_RESULTS_DIR
    / "selected_evidence_prototype.csv"
)


BASELINE_SELECTION_PATH = (
    SELECTOR_RESULTS_DIR
    / "usefulness_topk_baseline.csv"
)


PRODUCT_METRICS_PATH = (
    SELECTOR_RESULTS_DIR
    / "selector_product_metrics.csv"
)


COMPARISON_PRODUCT_PATH = (
    SELECTOR_RESULTS_DIR
    / "selector_baseline_comparison_by_product.csv"
)


COMPARISON_OVERALL_PATH = (
    SELECTOR_RESULTS_DIR
    / "selector_baseline_comparison_overall.csv"
)


CONFIG_PATH = (
    SELECTOR_RESULTS_DIR
    / "selector_config.json"
)


In [30]:
selected_evidence_df.to_csv(
    SELECTED_EVIDENCE_PATH,
    index=False,
)


usefulness_topk_df.to_csv(
    BASELINE_SELECTION_PATH,
    index=False,
)


proposed_product_metrics_df.to_csv(
    PRODUCT_METRICS_PATH,
    index=False,
)


selector_comparison_product_df.to_csv(
    COMPARISON_PRODUCT_PATH,
    index=False,
)


selector_comparison_overall_df.to_csv(
    COMPARISON_OVERALL_PATH,
    index=False,
)


selector_config = {
    "budget_k":
        SELECT_K,

    "weights": {
        "usefulness":
            WEIGHT_USEFULNESS,

        "representativeness":
            WEIGHT_REPRESENTATIVENESS,

        "coverage":
            WEIGHT_COVERAGE,

        "minority":
            WEIGHT_MINORITY,

        "redundancy_penalty":
            WEIGHT_REDUNDANCY,
    },

    "minority_priority_groups":
        MINORITY_PRIORITY_GROUPS,

    "minority_reserved_slots":
        MINORITY_RESERVED_SLOTS,

    "selector":
        "two_phase_constrained_greedy_prototype",

    "representativeness":
        "product_centroid_cosine_similarity",

    "redundancy":
        "maximum_cosine_similarity_to_selected",

    "opinion_fidelity":
        "jensen_shannon_divergence",

    "baseline":
        "predicted_usefulness_top_k",

    "mock_c1_warning":
        (
            "Aspect/sentiment and minority fields "
            "are synthetic integration-test placeholders."
        ),
}


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        selector_config,
        f,
        indent=2,
    )


print(
    "Selected evidence saved:",
    SELECTED_EVIDENCE_PATH
)


print(
    "Baseline selection saved:",
    BASELINE_SELECTION_PATH
)


print(
    "Product metrics saved:",
    PRODUCT_METRICS_PATH
)


print(
    "Per-product comparison saved:",
    COMPARISON_PRODUCT_PATH
)


print(
    "Overall comparison saved:",
    COMPARISON_OVERALL_PATH
)


print(
    "Selector config saved:",
    CONFIG_PATH
)


Selected evidence saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\selected_evidence_prototype.csv
Baseline selection saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\usefulness_topk_baseline.csv
Product metrics saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_product_metrics.csv
Per-product comparison saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_baseline_comparison_by_product.csv
Overall comparison saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_baseline_comparison_overall.csv
Selector config saved: d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_config.json


## 18. Final Prototype Validation

The final checks verify that:

- all products are represented,
- fixed-budget constraints are respected,
- no review is selected twice,
- selector scores are finite,
- expected output files were created.


In [31]:
assert (
    selected_evidence_df[
        "parent_asin"
    ].nunique()
    ==
    selector_ready_df[
        "parent_asin"
    ].nunique()
)


assert selected_evidence_df[
    "review_id"
].is_unique


assert (
    selected_evidence_df
    .groupby(
        "parent_asin"
    )
    .size()
    .le(
        SELECT_K
    )
    .all()
)


assert np.isfinite(
    selected_evidence_df[
        "selection_score"
    ]
).all()


assert SELECTOR_READY_PATH.exists()

assert SELECTOR_PREP_SUMMARY_PATH.exists()

assert SELECTED_EVIDENCE_PATH.exists()

assert BASELINE_SELECTION_PATH.exists()

assert PRODUCT_METRICS_PATH.exists()

assert COMPARISON_PRODUCT_PATH.exists()

assert COMPARISON_OVERALL_PATH.exists()

assert CONFIG_PATH.exists()


print(
    "Function 2 constrained-selector "
    "prototype validation passed."
)


Function 2 constrained-selector prototype validation passed.


## 19. Prototype Completion Summary

This notebook provides the first complete vertical slice of the Component 2 review-evidence selector:

**Review text → predicted usefulness → semantic representation → representativeness → C1-style aspect/sentiment evidence → supported-minority membership → redundancy-aware constrained greedy selection → fixed-budget review evidence set**

### Implemented

- Function 1 usefulness inference for candidate reviews
- within-product usefulness normalization
- stable Sentence-BERT embedding mapping
- semantic representativeness
- aspect coverage representation
- supported-minority membership
- semantic redundancy penalty
- fixed review budget
- budget-feasible priority minority groups
- minority-preservation reserve
- two-phase constrained greedy selection
- selected-evidence diagnostics
- aspect-coverage evaluation
- supported-minority recall
- priority-minority coverage
- semantic redundancy evaluation
- Jensen-Shannon opinion-fidelity diagnostic
- Predicted-Usefulness Top-K baseline comparison
- selector artifact and configuration persistence

### Current Limitations

- Final C1 aspect/sentiment outputs are not yet available.
- Current C1-derived fields are synthetic integration-test placeholders.
- Minority, aspect-coverage, and opinion-fidelity results are therefore not final research findings.
- Selector weights are provisional.
- `k = 5` is a prototype budget.
- The usefulness model is the preliminary TF-IDF + Ridge baseline.
- The selector prototype currently compares only against Predicted-Usefulness Top-K.
- Applying the saved Function 1 model to all candidate reviews is suitable for prototype integration, but final unbiased evaluation should use an appropriate held-out or out-of-fold scoring protocol.

### Later Research Work

The final study should include:

- real C1 aspect/sentiment outputs,
- broader smartphone and laptop data,
- Observed Helpful-Vote Top-K baseline,
- Predicted Helpfulness Top-K baseline,
- Helpfulness + MMR baseline,
- Aspect-Coverage baseline,
- Proposed selector without minority preservation,
- full proposed selector,
- removal-of-minority ablation,
- removal-of-redundancy ablation,
- selector-weight sensitivity analysis,
- review-budget sensitivity analysis,
- minority-threshold sensitivity analysis,
- bootstrap confidence intervals,
- Wilcoxon signed-rank tests where appropriate.

### Status

**First Function 2 constrained-selector prototype implemented and ready for end-to-end execution testing.**
